### Import et Configuration

In [39]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Configuration des axes cibles
AXES_CIBLES = ["Bd_Richard_Lenoir", "Quai_Conti", "Sts_Peres"]
FICHIER_CSV = 'data/raw/comptages.csv'

print("Configuration chargée")
print(f"Fichier cible : {FICHIER_CSV}")
print(f"Axes analysés : {AXES_CIBLES}")

Configuration chargée
Fichier cible : data/raw/comptages.csv
Axes analysés : ['Bd_Richard_Lenoir', 'Quai_Conti', 'Sts_Peres']


### Chargement

In [40]:
print("Chargement des données...")

# Chargement direct
df = pd.read_csv(FICHIER_CSV, sep=';')

# Filtrage pour ne garder que nos 3 axes
df = df[df['libelle'].isin(AXES_CIBLES)].copy()

print("Chargement terminé.")
print(f"Nombre total de lignes : {len(df)}")
print(f"Nombre de colonnes : {df.shape[1]}")

Chargement des données...
Chargement terminé.
Nombre total de lignes : 69824
Nombre de colonnes : 11


### 1. Format date en datetime et heure de paris, identifiant en texte(string) et etat en catégory
   Le but ici est de convertir la colonne 't_1h' en datetime avec le fuseau horaire de Paris


In [41]:
# 1. Conversion de la colonne 't_1h' en datetime avec le fuseau horaire de Paris
df['t_1h'] = pd.to_datetime(df['t_1h'], utc=True).dt.tz_convert('Europe/Paris')

# 2. Conversion des identifiants en texte (string)
colonnes_identifiants = ['iu_ac', 'iu_nd_amont', 'iu_nd_aval']
for col in colonnes_identifiants:
    df[col] = df[col].astype(str)

# 3. Conversion des colonnes d'état en catégorie
colonnes_etat = ['etat_trafic', 'etat_barre']
for col in colonnes_etat:
    df[col] = df[col].astype('category')

print("Types de données après transformation :")
print(df.dtypes)

Types de données après transformation :
iu_ac                                        str
libelle                                      str
t_1h                datetime64[us, Europe/Paris]
q                                        float64
k                                        float64
etat_trafic                             category
iu_nd_amont                                  str
libelle_nd_amont                             str
iu_nd_aval                                   str
libelle_nd_aval                              str
etat_barre                              category
dtype: object


### 2. Verification de doublons à l'aide de la bonne clé et si il y'en a on vas devoir les supprimer


In [42]:
# 1. Vérification des doublons globaux
nb_doublons_global = df.duplicated().sum()
print(f"Nombre de lignes strictement identiques (toutes colonnes) : {nb_doublons_global}")

# 2. Vérification des doublons sur la clé unique (tronçon + timestamp)
cle_unique = ['iu_ac', 't_1h']
nb_doublons_cle = df.duplicated(subset=cle_unique).sum()
print(f"Nombre de lignes en double sur la clé {cle_unique} : {nb_doublons_cle}")

Nombre de lignes strictement identiques (toutes colonnes) : 0
Nombre de lignes en double sur la clé ['iu_ac', 't_1h'] : 0


In [43]:
def groupes_identiques(df, col):
    """Associe à chaque capteur un numéro de groupe : même numéro = série identique."""
    large = df.pivot_table(index="t_1h", columns="iu_ac", values=col)
    signature = large.round(3).fillna(-1).T.apply(lambda s: hash(tuple(s)), axis=1)
    return signature.map({h: i for i, h in enumerate(signature.unique())})

groupe_q = groupes_identiques(df, "q")
groupe_k = groupes_identiques(df, "k")

print(f"Debit q      : {len(groupe_q)} capteurs, {groupe_q.nunique()} series differentes")
print(f"Occupation k : {len(groupe_k)} capteurs, {groupe_k.nunique()} series differentes")

# Identification du plus grand groupe de capteurs au débit identique
plus_grand = groupe_q.value_counts().index[0]
capteurs_identiques = groupe_q[groupe_q == plus_grand].index.tolist()

print(f"\nLe plus grand groupe de capteurs au debit identique (groupe {plus_grand}) contient {len(capteurs_identiques)} capteurs :")
print(capteurs_identiques)

Debit q      : 16 capteurs, 5 series differentes
Occupation k : 16 capteurs, 13 series differentes

Le plus grand groupe de capteurs au debit identique (groupe 0) contient 5 capteurs :
['1314', '1316', '1318', '1321', '410']


L'analyse révèle une forte redondance dans les données de débit (q) : sur 16 capteurs, seulement 5 séries temporelles distinctes sont identifiées. Un groupe de 5 capteurs (1314, 1316, 1318, 1321, 410) renvoie strictement les mêmes valeurs de débit.

In [44]:
test_amont = df.groupby('iu_nd_amont')['libelle_nd_amont'].nunique()
test_aval = df.groupby('iu_nd_aval')['libelle_nd_aval'].nunique()

print("--- Test de redondance ID <-> Libellé ---")
print(f"nœuds amont : {len(test_amont)} IDs uniques -> {test_amont.max()} libellés max par ID")
print(f"nœuds aval  : {len(test_aval)} IDs uniques -> {test_aval.max()} libellés max par ID")


--- Test de redondance ID <-> Libellé ---
nœuds amont : 12 IDs uniques -> 1 libellés max par ID
nœuds aval  : 12 IDs uniques -> 1 libellés max par ID


Relation 1-1 parfaite. Les libellés sont redondants avec les IDs.
On peut supprimer 'libelle_nd_amont' et 'libelle_nd_aval'

In [45]:
contingence = pd.crosstab(df['etat_trafic'], df['etat_barre'])
print("\n--- Tableau de contingence : etat_trafic vs etat_barre ---")
print(contingence)

total = len(df)
print(f"\nTotal d'observations : {total}")


--- Tableau de contingence : etat_trafic vs etat_barre ---
etat_barre   Invalide  Ouvert
etat_trafic                  
Bloqué              0      41
Fluide              0   65908
Inconnu          1861     136
Pré-saturé          0    1496
Saturé              0     382

Total d'observations : 69824



- Si 'Invalide' correspond toujours à 'Inconnu' -> redondance partielle
- Si 'Ouvert' correspond à plusieurs états de trafic -> pas de redondance, on garde les deux

In [46]:
# --- TEST 3 : Redondance entre libelle et iu_ac ---
# Est-ce que chaque capteur (iu_ac) appartient à un seul axe (libelle) ?

test_capteur_axe = df.groupby('iu_ac')['libelle'].nunique()
print("\n--- Test : un capteur = un seul axe ? ---")
print(f"Nombre de capteurs : {len(test_capteur_axe)}")
print(f"Nombre max d'axes par capteur : {test_capteur_axe.max()}")

if test_capteur_axe.max() == 1:
    print(" CONSTAT : Chaque capteur appartient à un seul axe.")
    


--- Test : un capteur = un seul axe ? ---
Nombre de capteurs : 16
Nombre max d'axes par capteur : 1
 CONSTAT : Chaque capteur appartient à un seul axe.


On constate que chaque capteur appartient à un seul axe.
   - 'libelle' est redondant avec 'iu_ac' (on peut retrouver l'axe via l'ID)
   -  MAIS on garde 'libelle' car c'est plus lisible pour l'analyse humaine

In [47]:
colonnes_a_supprimer = []

if test_amont.max() == 1 and test_aval.max() == 1:
    colonnes_a_supprimer.extend(['libelle_nd_amont', 'libelle_nd_aval'])
    print("Suppression de : libelle_nd_amont, libelle_nd_aval (redondants avec les IDs)")

if colonnes_a_supprimer:
    df = df.drop(columns=colonnes_a_supprimer)
    print(f"\nDataFrame après nettoyage : {df.shape[1]} colonnes")
else:
    print("\nAucune colonne redondante détectée. On conserve tout.")


Suppression de : libelle_nd_amont, libelle_nd_aval (redondants avec les IDs)

DataFrame après nettoyage : 9 colonnes


Après avoir supprimer libelle_nd_amont, libelle_nd_aval qui sont redondant avec les IDs il nous reste les colonnes ['iu_ac', 'libelle', 't_1h', 'q', 'k', 'etat_trafic', 'iu_nd_amont', 'iu_nd_aval', 'etat_barre']

### 4. Verification des valeurs manquantes : que veulent dire *Barré* et *Invalide* sur vos axes ? Quelle est la longueur de vos trous ? Choisissez votre méthode à partir de ces chiffres.

In [50]:
def analyser_etat(df, colonne_etat):
    print(f"--- Analyse de la colonne : {colonne_etat} ---")
    
    resume = df.groupby(colonne_etat).agg(
        nb_lignes=(colonne_etat, 'count'),
        q_moyen=('q', 'mean'),
        q_min=('q', 'min'),
        q_max=('q', 'max'),
        k_moyen=('k', 'mean')
    ).round(2)
    
    display(resume)
    print("\n")

analyser_etat(df, 'etat_trafic')
analyser_etat(df, 'etat_barre')

--- Analyse de la colonne : etat_trafic ---


,nb_lignes,q_moyen,q_min,q_max,k_moyen
etat_trafic,,,,,
Bloqué,41,752.07,138.0,942.0,55.34
Fluide,65908,338.50,0.0,1521.0,3.55
Inconnu,1997,394.09,39.0,879.0,NaN
Pré-saturé,1496,752.31,32.0,1563.0,20.51
Saturé,382,800.26,121.0,1496.0,37.04




--- Analyse de la colonne : etat_barre ---


,nb_lignes,q_moyen,q_min,q_max,k_moyen
etat_barre,,,,,
Invalide,1861,NaN,NaN,NaN,NaN
Ouvert,67963,350.59,0.0,1563.0,4.15


 1. Signification des états "Invalide" et "Inconnu au lieu de barré pour nos axes choisi" : Vérification par les données

**Cohérence physique (Fluide à Bloqué)** : Progression logique de l'occupation (k moyen passant de 3,5 % à 55,3 %) et du débit (q moyen de 338 à 752 véh/h), confirmant la réalité physique des embouteillages.

**États "Inconnu" et "Invalide"** : L'état Invalide affiche exclusivement des **NaN** (débit et occupation nuls). L'état Inconnu présente un débit moyen (394 véh/h) mais une occupation (k) manquante. Cela traduit des incertitudes de classification ou des défauts de mesure, et non un état de trafic exploitable.

**Conclusion** : Ces états ne représentent pas du trafic réel. Ils doivent être convertis en NaN et interpolés pour garantir la fiabilité des analyses ultérieures.

2. Longueurs des trous

In [10]:
# 1. Créer un masque combinant les types de "trous"
df['est_trou'] = (
    df['etat_trafic'].isin(['Inconnu']) |
    df['etat_barre'].isin(['Invalide']) |
    (df['q'] <= 5)
)

# 2. Trier par tronçon et par heure pour garantir la continuité temporelle
df_sorted = df.sort_values(by=['iu_ac', 't_1h']).copy()

# 3. Identifier les blocs consécutifs de trous
df_sorted['changement'] = df_sorted['est_trou'] != df_sorted['est_trou'].shift(1)
df_sorted['id_groupe'] = df_sorted['changement'].cumsum()

# 4. Isoler uniquement les lignes qui sont des trous
trous = df_sorted[df_sorted['est_trou']].copy()

if len(trous) == 0:
    print("Aucun trou détecté dans les données.")
else:
    resume_trous = trous.groupby(['iu_ac', 'id_groupe']).agg(
        debut=('t_1h', 'min'),
        fin=('t_1h', 'max'),
        duree_heures=('t_1h', lambda x: int((x.max() - x.min()).total_seconds() / 3600) + 1),
        nb_lignes=('est_trou', 'count')
    ).reset_index()

    resume_trous = resume_trous.sort_values(by=['iu_ac', 'duree_heures'], ascending=[True, False])

    print("--- Détail des trous de données (les 10 plus longs) ---")
    display(resume_trous.head(10))
    
    duree_moyenne = resume_trous['duree_heures'].mean()
    duree_mediane = resume_trous['duree_heures'].median()
    nb_trous_courts = (resume_trous['duree_heures'] <= 3).sum()
    nb_trous_longs = (resume_trous['duree_heures'] > 3).sum()
    
    print(f"\n--- Statistiques globales ---")
    print(f"Nombre total de trous identifiés : {len(resume_trous)}")
    print(f"Durée médiane d'un trou : {duree_mediane:.1f} heures")
    print(f"\nRépartition :")
    print(f"  - Trous courts (<= 3h) : {nb_trous_courts} ({nb_trous_courts/len(resume_trous)*100:.1f}%)")
    print(f"  - Trous longs (> 3h) : {nb_trous_longs} ({nb_trous_longs/len(resume_trous)*100:.1f}%)")

--- Détail des trous de données (les 10 plus longs) ---


,iu_ac,id_groupe,debut,fin,duree_heures,nb_lignes
13,1314,28,2026-03-27 11:00:00+01:00,2026-04-02 11:00:00+02:00,144,144
14,1314,30,2026-04-14 01:00:00+02:00,2026-04-14 11:00:00+02:00,11,8
10,1314,22,2026-03-23 11:00:00+01:00,2026-03-23 19:00:00+01:00,9,9
20,1314,42,2026-06-03 13:00:00+02:00,2026-06-03 17:00:00+02:00,5,5
11,1314,24,2026-03-23 22:00:00+01:00,2026-03-24 00:00:00+01:00,3,3
21,1314,44,2026-06-07 13:00:00+02:00,2026-06-07 15:00:00+02:00,3,3
5,1314,12,2026-03-19 20:00:00+01:00,2026-03-19 21:00:00+01:00,2,2
23,1314,48,2026-07-22 09:00:00+02:00,2026-07-22 10:00:00+02:00,2,2
26,1314,54,2026-08-03 10:00:00+02:00,2026-08-03 11:00:00+02:00,2,2
0,1314,2,2026-03-12 12:00:00+01:00,2026-03-12 12:00:00+01:00,1,1



--- Statistiques globales ---
Nombre total de trous identifiés : 451
Durée médiane d'un trou : 1.0 heures

Répartition :
  - Trous courts (<= 3h) : 405 (89.8%)
  - Trous longs (> 3h) : 46 (10.2%)


In [51]:
print("Début du traitement des valeurs manquantes...")

# 1. Identifier les lignes problématiques et les mettre à NaN
masque_trou = (
    df['etat_trafic'].isin(['Inconnu']) |
    df['etat_barre'].isin(['Invalide']) |
    (df['q'] <= 5)
)

df.loc[masque_trou, 'q'] = np.nan
df.loc[masque_trou, 'k'] = np.nan

nb_nan_initial = masque_trou.sum()
print(f"Nombre de valeurs mises à NaN : {nb_nan_initial}")

# 2. Appliquer l'interpolation linéaire PAR TRONÇON (iu_ac) avec limite de 3 heures

df['q'] = df.groupby('iu_ac')['q'].transform(
    lambda x: x.interpolate(method='linear', limit=3, limit_direction='both')
)
df['k'] = df.groupby('iu_ac')['k'].transform(
    lambda x: x.interpolate(method='linear', limit=3, limit_direction='both')
)

# 3. Vérification finale
nb_nan_final_q = df['q'].isna().sum()
nb_interpole = nb_nan_initial - nb_nan_final_q

print(f"\nTraitement terminé.")
print(f"Valeurs manquantes initiales : {nb_nan_initial}")
print(f"Valeurs interpolées (trous <= 3h) : {nb_interpole}")
print(f"Valeurs manquantes restantes (trous > 3h) : {nb_nan_final_q}")
print(f"Taux d'interpolation : {nb_interpole/nb_nan_initial*100:.1f}%")

Début du traitement des valeurs manquantes...
Nombre de valeurs mises à NaN : 1999

Traitement terminé.
Valeurs manquantes initiales : 1999
Valeurs interpolées (trous <= 3h) : 496
Valeurs manquantes restantes (trous > 3h) : 1503
Taux d'interpolation : 24.8%


### Statistiques par tronçon avec heures de pointe

In [52]:
# Extraction de l'heure pour identifier les heures de pointe
df['heure'] = df['t_1h'].dt.hour

# Filtrage des heures de pointe classiques (8h-10h et 17h-19h)
heures_pointe = df[df['heure'].isin([8, 9, 17, 18])]

# Calcul des statistiques par tronçon (iu_ac)
stats_troncon_pointe = heures_pointe.groupby('iu_ac').agg({
    'q': ['mean', 'max'],
    'k': 'mean'
}).round(2)

# Renommage des colonnes pour la lisibilité
stats_troncon_pointe.columns = ['Debit_moyen', 'Debit_max', 'Occupation_moyenne']

print("--- Statistiques par tronçon durant les heures de pointe ---")
display(stats_troncon_pointe.sort_values('Debit_moyen', ascending=False))

--- Statistiques par tronçon durant les heures de pointe ---


,Debit_moyen,Debit_max,Occupation_moyenne
iu_ac,,,
70,1088.33,1563.0,10.52
69,1088.33,1563.0,10.52
192,504.01,910.0,14.56
191,503.86,910.0,7.24
1321,393.20,759.0,3.47
410,393.20,759.0,4.35
1318,393.13,759.0,2.55
1314,393.13,759.0,2.55
1316,393.07,759.0,5.32


In [53]:
# 1. Créer le dossier data/processed s'il n'existe pas
os.makedirs('data/processed', exist_ok=True)

# 2. Sauvegarder le DataFrame nettoyé
df.to_csv('data/processed/comptages_propres.csv', index=False, sep=';')

print("Données propres sauvegardées dans data/processed/comptages_propres.csv")
print(f"Nombre de lignes : {len(df)}")
print(f"Nombre de colonnes : {df.shape[1]}")

Données propres sauvegardées dans data/processed/comptages_propres.csv
Nombre de lignes : 69824
Nombre de colonnes : 10
